In [4]:
# win_predictor_app_pro.py

import streamlit as st
import pandas as pd
import numpy as np
import joblib
from PIL import Image
import requests
from io import BytesIO
import plotly.express as px

# ---------- PAGE CONFIG ----------
st.set_page_config(
    page_title="🏏 IPL Win Predictor PRO",
    page_icon="🏆",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.title("🏏 IPL Match Win Probability Predictor — PRO")
st.markdown("""
Predict the probability of the **second innings team winning** based on first innings stats.
""")

# ---------- LOAD MODEL & FEATURES ----------
model = joblib.load("../models/best_winprob_model.pkl")
x_columns = joblib.load("../models/x_columns.pkl")  # must match ML training

# ---------- TEAM DATA ----------
top_teams = [
    'Mumbai Indians','Chennai Super Kings','Kolkata Knight Riders',
    'Royal Challengers Bangalore','Kings XI Punjab','Rajasthan Royals',
    'Delhi Daredevils','Sunrisers Hyderabad','Deccan Chargers',
    'Pune Warriors','Delhi Capitals','Gujarat Lions'
]

team_logos = {
    'Mumbai Indians': 'https://i.ibb.co/7VwXrVv/MI.png',
    'Chennai Super Kings': 'https://i.ibb.co/jTG6Wy9/CSK.png',
    'Kolkata Knight Riders': 'https://i.ibb.co/yYPXfnv/KKR.png',
    'Royal Challengers Bangalore': 'https://i.ibb.co/m85G9Q9/RCB.png',
    'Kings XI Punjab': 'https://i.ibb.co/YWq8cXJ/KXIP.png',
    'Rajasthan Royals': 'https://i.ibb.co/CBx6m3y/RR.png',
    'Delhi Daredevils': 'https://i.ibb.co/6vws8R0/DD.png',
    'Sunrisers Hyderabad': 'https://i.ibb.co/d6t7MZp/SRH.png',
    'Deccan Chargers': 'https://i.ibb.co/w6qR1Mt/DC.png',
    'Pune Warriors': 'https://i.ibb.co/7J5Y2k2/PW.png',
    'Delhi Capitals': 'https://i.ibb.co/9ytzFZp/DCap.png',
    'Gujarat Lions': 'https://i.ibb.co/0B8FQz6/GL.png'
}

# ---------- SIDEBAR INPUTS ----------
st.sidebar.header("Match Inputs")
batting_team = st.sidebar.selectbox("Batting Team (Innings 1)", top_teams)
bowling_team = st.sidebar.selectbox("Bowling Team (Innings 2)", top_teams)

st.sidebar.subheader("First Innings Runs")
pp_runs = st.sidebar.slider("Powerplay (0-5 overs)", 0, 100, 40)
mid_runs = st.sidebar.slider("Middle Overs (6-15)", 0, 150, 70)
death_runs = st.sidebar.slider("Death Overs (16-20)", 0, 100, 50)

st.sidebar.subheader("First Innings Wickets")
pp_wickets = st.sidebar.slider("Powerplay Wickets", 0, 6, 1)
mid_wickets = st.sidebar.slider("Middle Wickets", 0, 10, 2)
death_wickets = st.sidebar.slider("Death Wickets", 0, 10, 2)

extras = st.sidebar.slider("Extras Runs", 0, 20, 2)

# ---------- PREPARE INPUT DATA ----------
input_dict = {
    'pp_runs': pp_runs,
    'mid_runs': mid_runs,
    'death_runs': death_runs,
    'pp_wickets': pp_wickets,
    'mid_wickets': mid_wickets,
    'death_wickets': death_wickets,
    'extras': extras
}

# Encode teams
for team in top_teams:
    input_dict[f'team1_{team}'] = 1 if batting_team == team else 0
    input_dict[f'team2_{team}'] = 1 if bowling_team == team else 0

for col in x_columns:
    if col not in input_dict:
        input_dict[col] = 0

input_df = pd.DataFrame([input_dict])[x_columns]

# ---------- PREDICTION ----------
prob = model.predict_proba(input_df)[0][1]

# ---------- DYNAMIC BACKGROUND ----------
if prob >= 0.7:
    st.markdown(
        f"<div style='padding:10px;border-radius:10px;background-color:#4CAF50;color:white'>High chance to win: {prob*100:.2f}%</div>",
        unsafe_allow_html=True
    )
elif prob >= 0.4:
    st.markdown(
        f"<div style='padding:10px;border-radius:10px;background-color:#FFC107;color:black'>Moderate chance: {prob*100:.2f}%</div>",
        unsafe_allow_html=True
    )
else:
    st.markdown(
        f"<div style='padding:10px;border-radius:10px;background-color:#F44336;color:white'>Low chance: {prob*100:.2f}%</div>",
        unsafe_allow_html=True
    )

# ---------- LAYOUT: TEAMS + LOGOS ----------
col1, col2, col3 = st.columns([1,2,1])
with col1:
    st.image(team_logos[batting_team], width=120)
    st.markdown(f"**{batting_team}**")
with col2:
    st.subheader("VS")
with col3:
    st.image(team_logos[bowling_team], width=120)
    st.markdown(f"**{bowling_team}**")

# ---------- METRIC CARDS ----------
col1, col2, col3 = st.columns(3)
col1.metric("Powerplay Runs", pp_runs)
col2.metric("Middle Overs Runs", mid_runs)
col3.metric("Death Overs Runs", death_runs)

col4, col5, col6 = st.columns(3)
col4.metric("Powerplay Wickets", pp_wickets)
col5.metric("Middle Wickets", mid_wickets)
col6.metric("Death Wickets", death_wickets)

# ---------- RUN BREAKDOWN CHART ----------
st.subheader("📊 First Innings Runs Breakdown")
run_data = pd.DataFrame({
    'Phase': ['Powerplay', 'Middle', 'Death'],
    'Runs': [pp_runs, mid_runs, death_runs]
})
fig = px.bar(run_data, x='Phase', y='Runs', text='Runs', color='Phase',
             color_discrete_map={'Powerplay':'#4CAF50','Middle':'#FFC107','Death':'#F44336'})
st.plotly_chart(fig, use_container_width=True)

# ---------- WICKETS IMPACT SIMULATION ----------
st.subheader("🎯 Wickets Impact Simulation")
sim_wickets = np.arange(0, max(pp_wickets+mid_wickets+death_wickets+1, 11))
sim_probs = []

for w in sim_wickets:
    tmp_dict = input_dict.copy()
    tmp_dict['pp_wickets'] = min(pp_wickets, w)
    tmp_dict['mid_wickets'] = min(mid_wickets, w-pp_wickets if w-pp_wickets>0 else 0)
    tmp_dict['death_wickets'] = min(death_wickets, max(0, w-pp_wickets-mid_wickets))
    tmp_df = pd.DataFrame([tmp_dict])[x_columns]
    sim_probs.append(model.predict_proba(tmp_df)[0][1]*100)

sim_df = pd.DataFrame({'Total Wickets': sim_wickets, 'Win Probability (%)': sim_probs})
fig2 = px.line(sim_df, x='Total Wickets', y='Win Probability (%)', markers=True)
st.plotly_chart(fig2, use_container_width=True)

# ---------- FOOTER ----------
st.markdown("""
---
**Instructions:**  
- Use the sidebar to select teams and enter match stats.  
- Probability updates dynamically based on your inputs.  
- Explore how losing wickets affects win probability.  
""")


2025-11-19 19:11:27.118 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-19 19:11:27.118 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-19 19:11:27.118 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-19 19:11:27.124 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-19 19:11:27.126 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-19 19:11:27.127 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-19 19:11:27.127 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-19 19:11:27.136 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()